# Pertemuan 2 — Data Wrangling & EDA dengan Python (pandas)
**Business Statistics and Data Visualization · Prodi S1 Bisnis Digital**

Notebook hands-on ini memandu Anda membersihkan dan mengeksplorasi data transaksi toko online fiktif **TokoKita** (1.537 baris, Agustus–September 2026).

**Tujuan (Sub-CPMK-2):** mahasiswa mampu melakukan pembersihan dan penyiapan data menggunakan spreadsheet dan Python.

| Bagian | Isi | Perkiraan waktu |
|---|---|---|
| 0 | Persiapan & memuat data | 10 menit |
| 1 | Mengenali data (*inspect*) | 20 menit |
| 2 | Membersihkan data: duplikat, tipe data, label, missing value, nilai tidak valid, outlier | 70 menit |
| 3 | Transformasi: kolom turunan | 20 menit |
| 4 | Exploratory Data Analysis (EDA) | 35 menit |
| 5 | Menyimpan data bersih & *cleaning log* | 5 menit |
| 6 | Tugas 2 (5 soal) | dikerjakan di rumah |

**Cara pakai di Google Colab**
1. Buka [colab.research.google.com](https://colab.research.google.com) → *File → Upload notebook* → pilih file ini.
2. Jalankan sel satu per satu dengan **Shift + Enter**.
3. Sel bertanda 🧩 **Coba sendiri** berisi latihan singkat. Isi kodenya lalu jalankan.

> Semua data dalam notebook ini fiktif dan dibuat khusus untuk latihan.

## 0. Persiapan

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette=["#0FA3B1", "#16213E", "#F7B32B", "#EE6C4D", "#5B6770"])
plt.rcParams["figure.figsize"] = (9, 4.5)
print("pandas", pd.__version__)

### Memuat data
Unduh `transaksi_tokokita_kotor.csv` dari LMS. Di Colab, sel di bawah akan meminta Anda mengunggah file jika belum ada.

In [ ]:
FILE = "transaksi_tokokita_kotor.csv"

if not os.path.exists(FILE):
    try:
        from google.colab import files   # hanya tersedia di Colab
        print("Silakan unggah file", FILE)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Letakkan {FILE} di folder yang sama dengan notebook ini.")

df_raw = pd.read_csv(FILE)
df = df_raw.copy()          # selalu bekerja pada salinan; data mentah tidak diubah
df.head(10)

## 1. Mengenali data (*inspect*)
Sebelum membersihkan apa pun, jawab dulu lima pertanyaan: **berapa ukurannya, apa saja kolomnya, apa tipe tiap kolom, adakah yang kosong, dan seperti apa sebaran nilainya?**

In [ ]:
print("Ukuran data (baris, kolom):", df.shape)
df.info()

**Perhatikan:** kolom `harga` dan `tanggal` bertipe `object` atau `str` (teks), padahal seharusnya angka dan tanggal. Kolom `kota`, `metode_bayar`, dan `rating` memiliki *non-null count* lebih kecil dari jumlah baris, artinya ada data kosong.

In [ ]:
df.describe(include="all").T

In [ ]:
# Nilai unik pada kolom kategorik: di sinilah label yang tidak konsisten terlihat
for kol in ["kota", "metode_bayar", "kategori"]:
    print(f"\n=== {kol} ({df[kol].nunique()} nilai unik) ===")
    print(df[kol].value_counts(dropna=False).to_string())

### 🧩 Coba sendiri 1
Tampilkan sebaran nilai kolom `rating` (termasuk yang kosong). Nilai apa saja yang **tidak masuk akal** untuk rating bintang 1–5?

In [ ]:
# Tulis kode Anda di sini. Petunjuk: gunakan .value_counts(dropna=False).sort_index()

## 2. Membersihkan data
Kita catat setiap langkah di **cleaning log** agar proses dapat diaudit dan diulang. Ini kebiasaan penting analis profesional.

In [ ]:
cleaning_log = []

def catat(langkah, keterangan, n_sebelum, n_sesudah):
    cleaning_log.append({"langkah": langkah, "keterangan": keterangan,
                         "baris_sebelum": n_sebelum, "baris_sesudah": n_sesudah,
                         "baris_terdampak": n_sebelum - n_sesudah})

def catat_ubah(langkah, keterangan, n_terdampak):
    # untuk langkah yang mengubah nilai tanpa menghapus baris
    cleaning_log.append({"langkah": langkah, "keterangan": keterangan, "baris_sebelum": len(df),
                         "baris_sesudah": len(df), "baris_terdampak": int(n_terdampak)})

### 2.1 Duplikat
Ada dua jenis duplikat:
- **Duplikat persis**: seluruh kolom sama (biasanya akibat data terunggah dua kali).
- **Duplikat kunci**: `order_id` sama tetapi isinya berbeda. Ini perlu diselidiki, bukan langsung dihapus.

In [ ]:
print("Duplikat persis      :", df.duplicated().sum())
print("Duplikat order_id    :", df["order_id"].duplicated().sum())

# Contoh pasangan duplikat
contoh = df[df.duplicated(keep=False)].sort_values("order_id").head(6)
contoh

In [ ]:
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
catat("Hapus duplikat", "Baris identik di semua kolom", n0, len(df))
print(f"Dihapus {n0 - len(df)} baris; sisa {len(df)} baris")
print("Sisa duplikat order_id:", df["order_id"].duplicated().sum())

### 2.2 Memperbaiki tipe data
**Harga** tersimpan dalam dua bentuk: `149500` dan `Rp 149.500`. Kita buang semua karakter selain angka, lalu ubah menjadi bilangan.

**Tanggal** juga punya dua format: `2026-08-15` (ISO) dan `15/08/2026` (hari/bulan/tahun). Kita parsing masing-masing format secara eksplisit agar tidak tertukar antara hari dan bulan.

In [ ]:
print("Contoh harga berbentuk teks:", df.loc[df["harga"].str.contains("Rp"), "harga"].head(3).tolist())

n_teks = df["harga"].str.contains("Rp").sum()
df["harga"] = (df["harga"]
               .str.replace(r"[^0-9]", "", regex=True)   # hapus 'Rp', spasi, titik
               .astype("int64"))
catat_ubah("Konversi harga ke angka", "format 'Rp 149.500' → 149500", n_teks)
df["harga"].describe()

In [ ]:
t_iso = pd.to_datetime(df["tanggal"], format="%Y-%m-%d", errors="coerce")
t_dmy = pd.to_datetime(df["tanggal"], format="%d/%m/%Y", errors="coerce")
print("Format ISO:", t_iso.notna().sum(), "| format dd/mm/yyyy:", t_dmy.notna().sum())

df["tanggal"] = t_iso.fillna(t_dmy)
catat_ubah("Konversi tanggal", "format dd/mm/yyyy diseragamkan", t_dmy.notna().sum())
print("Gagal diparsing:", df["tanggal"].isna().sum())
print("Rentang tanggal:", df["tanggal"].min().date(), "s.d.", df["tanggal"].max().date())

### 2.3 Menyeragamkan label kategori
`mks`, `MAKASSAR`, dan `Makassar ` (dengan spasi) adalah kota yang sama. Langkahnya: (1) hapus spasi di tepi, (2) samakan huruf, (3) petakan singkatan.

In [ ]:
sebelum = df["kota"].nunique()
kota_lama = df["kota"].copy()
peta_kota = {"Mks": "Makassar", "Jkt": "Jakarta", "Dki Jakarta": "Jakarta", "Sby": "Surabaya", "Bdg": "Bandung"}
df["kota"] = df["kota"].str.strip().str.title().replace(peta_kota)
print(f"Nilai unik kota: {sebelum} → {df['kota'].nunique()}")
catat_ubah("Seragamkan label kota", f"{sebelum} → {df['kota'].nunique()} nilai unik", (kota_lama != df["kota"]).sum() - kota_lama.isna().sum())
df["kota"].value_counts(dropna=False)

### 🧩 Coba sendiri 2
Seragamkan kolom `metode_bayar` sehingga hanya berisi empat nilai: `E-Wallet`, `COD`, `Transfer Bank`, `Kartu Kredit`. Kode contoh sudah disediakan. Periksa hasilnya dengan `value_counts()`.

In [ ]:
bayar_lama = df["metode_bayar"].copy()
peta_bayar = {"ewallet": "E-Wallet", "e-wallet": "E-Wallet", "cod": "COD",
              "transfer": "Transfer Bank", "transfer bank": "Transfer Bank",
              "kartu kredit": "Kartu Kredit", "cc": "Kartu Kredit"}
df["metode_bayar"] = df["metode_bayar"].str.strip().str.lower().replace(peta_bayar)
catat_ubah("Seragamkan metode bayar", f"{bayar_lama.nunique()} → {df['metode_bayar'].nunique()} nilai unik", (bayar_lama != df["metode_bayar"]).sum() - bayar_lama.isna().sum())
df["metode_bayar"].value_counts(dropna=False)

### 2.4 Missing value
Strategi menangani data kosong **bergantung pada penyebab dan kegunaannya**:

| Situasi | Strategi |
|---|---|
| Sedikit (< 5%) dan acak | hapus baris, atau isi kategori "Tidak diketahui" |
| Numerik, perlu dipakai | isi median kelompok (*imputasi*) + tandai dengan kolom *flag* |
| Kosong karena memang tidak ada (mis. pembeli tidak memberi rating) | **biarkan kosong**; analisis hanya pada data yang ada |

In [ ]:
miss = df.isna().sum().to_frame("jumlah_kosong")
miss["persen"] = (miss["jumlah_kosong"] / len(df) * 100).round(1)
miss = miss[miss["jumlah_kosong"] > 0].sort_values("persen")
ax = miss["persen"].plot.barh(color="#EE6C4D", figsize=(7, 2.6), title="Persentase data kosong per kolom")
ax.set_xlabel("% baris")
plt.show()
miss

In [ ]:
catat_ubah("Isi kota kosong", "diisi 'Tidak diketahui'", df["kota"].isna().sum())
catat_ubah("Isi metode bayar kosong", "diisi 'Tidak diketahui'", df["metode_bayar"].isna().sum())
# kota & metode_bayar: kategori, sedikit → isi "Tidak diketahui" agar baris (dan omzetnya) tidak hilang
df["kota"] = df["kota"].fillna("Tidak diketahui")
df["metode_bayar"] = df["metode_bayar"].fillna("Tidak diketahui")

# rating: kosong berarti pembeli tidak memberi ulasan → dibiarkan NaN, tapi ditandai
df["ada_rating"] = df["rating"].notna()
print(df[["kota", "metode_bayar", "rating"]].isna().sum())

### 2.5 Nilai tidak valid
Aturan validasi bisnis: **rating** harus 1–5 dan **jumlah** harus ≥ 1. Nilai di luar aturan adalah kesalahan input, bukan outlier.

In [ ]:
rating_salah = ~df["rating"].between(1, 5) & df["rating"].notna()
print("Rating di luar 1–5:", rating_salah.sum(), "→", sorted(df.loc[rating_salah, "rating"].unique()))
df.loc[rating_salah, "rating"] = np.nan          # jadikan kosong, baris tetap dipakai
catat_ubah("Rating tidak valid → kosong", "rating di luar 1–5", rating_salah.sum())
df["ada_rating"] = df["rating"].notna()

n0 = len(df)
print("Jumlah ≤ 0:", (df["jumlah"] <= 0).sum())
df = df[df["jumlah"] >= 1].reset_index(drop=True)  # transaksi tidak sah → dihapus
catat("Hapus jumlah tidak valid", "jumlah ≤ 0 (kesalahan input)", n0, len(df))

### 2.6 Outlier
Outlier adalah nilai yang jauh dari mayoritas. Metode **IQR**: nilai dianggap outlier bila berada di luar

$$[\,Q_1 - 1{,}5\times IQR,\; Q_3 + 1{,}5\times IQR\,], \quad IQR = Q_3 - Q_1$$

Outlier **tidak otomatis dihapus**. Tanyakan: apakah ini **kesalahan** (mis. salah ketik satu nol) atau **nyata** (mis. pembeli grosir)?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x="harga", y="kategori", ax=ax, color="#0FA3B1", flierprops=dict(markerfacecolor="#EE6C4D"))
ax.set_title("Harga per kategori: titik oranye = outlier menurut IQR")
ax.set_xlabel("Harga (Rp)")
plt.show()

In [ ]:
# Bandingkan setiap harga dengan median kategorinya
df["rasio_median"] = df["harga"] / df.groupby("kategori")["harga"].transform("median")
curiga = df[df["rasio_median"] > 5].sort_values("rasio_median", ascending=False)
curiga[["order_id", "kategori", "harga", "rasio_median"]]

Harga-harga di atas **lebih dari 5× median kategorinya**, padahal sebagian besar harga wajar berada di bawah 4,5× median. Untuk produk fashion, kecantikan, dan makanan, ini hampir pasti **salah ketik satu angka nol**. Keputusan: koreksi dengan membagi 10, lalu catat di cleaning log. Nilai di zona abu-abu (sekitar 4,5–5×) dibiarkan. Dalam praktik nyata, semua kasus seperti ini dikonfirmasi dulu ke tim penjualan.

In [ ]:
typo = (df["rasio_median"] > 5) & df["kategori"].isin(["Fashion", "Kecantikan", "Makanan"])
df.loc[typo, "harga"] = df.loc[typo, "harga"] // 10
catat_ubah("Koreksi harga salah ketik", "> 5× median kategori, dibagi 10", typo.sum())
df = df.drop(columns="rasio_median")
print("Harga dikoreksi:", typo.sum(), "baris")

In [ ]:
# Outlier pada jumlah: pembelian grosir. Ini transaksi nyata → dipertahankan, tetapi ditandai
q1, q3 = df["jumlah"].quantile([0.25, 0.75])
batas_atas = q3 + 1.5 * (q3 - q1)
df["grosir"] = df["jumlah"] > 20
catat_ubah("Tandai transaksi grosir", "jumlah > 20 unit; dipertahankan", df["grosir"].sum())
print(f"Batas atas IQR jumlah = {batas_atas:.1f}; transaksi grosir (>20 unit): {df['grosir'].sum()}")
df.loc[df["grosir"], ["order_id", "kategori", "harga", "jumlah", "kota"]]

## 3. Transformasi: membuat kolom turunan
Kolom baru membantu analisis: **total belanja**, informasi waktu, dan pengelompokan nilai belanja.

In [ ]:
nama_hari = {0: "Senin", 1: "Selasa", 2: "Rabu", 3: "Kamis", 4: "Jumat", 5: "Sabtu", 6: "Minggu"}

df["total"] = df["harga"] * df["jumlah"]
df["bulan"] = df["tanggal"].dt.month.map({8: "Agustus", 9: "September"})
df["hari"] = df["tanggal"].dt.dayofweek.map(nama_hari)
df["akhir_pekan"] = df["tanggal"].dt.dayofweek >= 5
df["kelompok_belanja"] = pd.cut(df["total"], bins=[0, 100_000, 300_000, 1_000_000, np.inf],
                                labels=["< 100 rb", "100–300 rb", "300 rb–1 jt", "> 1 jt"])
df["log_total"] = np.log10(df["total"])
df[["order_id", "tanggal", "hari", "harga", "jumlah", "total", "kelompok_belanja"]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(df["total"] / 1000, bins=40, color="#16213E")
axes[0].set_title("Total belanja: sangat menceng ke kanan"); axes[0].set_xlabel("ribu Rp")
axes[1].hist(df["log_total"], bins=40, color="#0FA3B1")
axes[1].set_title("Setelah transformasi log10: lebih simetris"); axes[1].set_xlabel("log10(total)")
plt.tight_layout(); plt.show()

## 4. Exploratory Data Analysis (EDA)
EDA = mengajukan pertanyaan ke data lewat ringkasan dan grafik. Mulai dari **satu variabel** (univariat), lalu **hubungan antarvariabel** (bivariat), lalu **per kelompok**.

In [ ]:
print(f"Jumlah transaksi : {len(df):,}")
print(f"Total omzet      : Rp{df['total'].sum():,.0f}")
print(f"Rata-rata belanja: Rp{df['total'].mean():,.0f} | median: Rp{df['total'].median():,.0f}")
print(f"Pelanggan unik   : {df['id_pelanggan'].nunique():,}")
print(f"Rating rata-rata : {df['rating'].mean():.2f} (dari {df['ada_rating'].sum():,} ulasan)")

In [ ]:
ringkas = (df.groupby("kategori")
             .agg(transaksi=("order_id", "count"), omzet=("total", "sum"),
                  median_belanja=("total", "median"), rating=("rating", "mean"))
             .sort_values("omzet", ascending=False))
ringkas["porsi_omzet_%"] = (ringkas["omzet"] / ringkas["omzet"].sum() * 100).round(1)
ringkas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(ringkas["omzet"] / 1e6).sort_values().plot.barh(ax=axes[0], color="#0FA3B1")
axes[0].set_title("Omzet per kategori (juta Rp)"); axes[0].set_ylabel("")
kota_omzet = df[df["kota"] != "Tidak diketahui"].groupby("kota")["total"].sum().sort_values() / 1e6
kota_omzet.plot.barh(ax=axes[1], color="#16213E")
axes[1].set_title("Omzet per kota (juta Rp)"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
harian = df.groupby("tanggal").agg(transaksi=("order_id", "count"), omzet=("total", "sum"))
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(harian.index, harian["transaksi"], color="#5B6770", lw=1, alpha=0.6, label="harian")
ax.plot(harian.index, harian["transaksi"].rolling(7).mean(), color="#EE6C4D", lw=2.5, label="rata-rata 7 hari")
ax.set_title("Jumlah transaksi per hari"); ax.legend(); plt.show()

In [ ]:
urutan = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
per_hari = df.groupby("hari")["order_id"].count().reindex(urutan) / df.groupby("hari")["tanggal"].nunique().reindex(urutan)
ax = per_hari.plot.bar(color=["#0FA3B1"] * 5 + ["#EE6C4D"] * 2, figsize=(8, 3.5), rot=0,
                       title="Rata-rata transaksi per hari dalam seminggu")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
rating_bayar = df.groupby("metode_bayar")["rating"].mean().drop("Tidak diketahui").sort_values()
rating_bayar.plot.barh(ax=ax, color="#F7B32B"); ax.set_xlim(3, 4.6)
ax.set_title("Rata-rata rating menurut metode bayar"); ax.set_ylabel("")
plt.show()
rating_bayar

In [ ]:
pivot = pd.pivot_table(df[df["kota"] != "Tidak diketahui"], index="kota", columns="kategori",
                       values="total", aggfunc="sum") / 1e6
plt.figure(figsize=(9, 3.8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "juta Rp"})
plt.title("Omzet kota × kategori (juta Rp)"); plt.ylabel(""); plt.xlabel("")
plt.show()

### Menulis insight
Insight yang baik = **temuan + angka + implikasi bisnis**. Contoh format:

> *Fashion menyumbang X% omzet, tetapi Elektronik memiliki median belanja Y kali lebih besar. → Kampanye bundling aksesori elektronik berpotensi menaikkan nilai keranjang.*

### 🧩 Coba sendiri 3
Tulis **dua insight** dari grafik di atas pada sel teks di bawah ini.

**Insight 1:** …

**Insight 2:** …

## 5. Menyimpan data bersih & cleaning log

In [ ]:
log = pd.DataFrame(cleaning_log)
print(f"Data mentah: {len(df_raw):,} baris → data bersih: {len(df):,} baris")
log

In [ ]:
df.to_csv("transaksi_tokokita_bersih.csv", index=False)
log.to_csv("cleaning_log.csv", index=False)
print("Tersimpan: transaksi_tokokita_bersih.csv & cleaning_log.csv")
# Di Colab, unduh dengan: from google.colab import files; files.download("transaksi_tokokita_bersih.csv")

---
## 6. Tugas 2 — Laporan Praktikum Pembersihan Data Transaksi
**Individu · bobot 3% nilai akhir · kumpulkan sebelum Pertemuan 3 melalui LMS** (file `.ipynb` + PDF hasil *File → Print*).

Gunakan dataset **`transaksi_tugas2_kotor.csv`** (data fiktif toko online, Juli–September 2026, 10 kolom, termasuk kolom `ongkir`). Dataset ini **berbeda** dari data hands-on: masalahnya mirip, tetapi angkanya lain.

| No | Soal | Bobot |
|---|---|---|
| 1 | **Profil data.** Laporkan ukuran data, tipe tiap kolom, dan jumlah serta persentase data kosong per kolom. Sebutkan kolom yang tipenya salah dan jelaskan mengapa. | 15% |
| 2 | **Duplikat & label.** Hapus duplikat persis, lalu seragamkan `kota` dan `metode_bayar`. Laporkan jumlah baris yang dihapus serta daftar nilai unik sebelum dan sesudah. | 20% |
| 3 | **Tipe data & missing value.** Ubah `harga` menjadi angka dan `tanggal` menjadi tipe tanggal. Tangani data kosong di setiap kolom dan **jelaskan alasan** strategi yang dipilih. | 20% |
| 4 | **Nilai tidak valid & outlier.** Terapkan aturan validasi untuk `rating` dan `jumlah`. Deteksi outlier `harga` dengan metode IQR per kategori, tentukan mana yang **kesalahan** dan mana yang **nyata**, lalu jelaskan tindakan Anda. | 25% |
| 5 | **EDA & insight.** Buat kolom `total = harga × jumlah + ongkir`, buat **tiga visualisasi** yang berbeda jenis, tulis **tiga insight bisnis** (temuan + angka + implikasi), dan tampilkan *cleaning log* lengkap. | 20% |

**Kriteria penilaian umum:** kode berjalan tanpa error dari awal sampai akhir, setiap keputusan pembersihan dijelaskan dalam sel teks, dan angka yang dilaporkan sesuai dengan output kode.

### Soal 1 — Profil data

In [ ]:
tugas = pd.read_csv("transaksi_tugas2_kotor.csv")
# Lanjutkan jawaban Soal 1 di sini

**Penjelasan:** …

### Soal 2 — Duplikat & label

In [ ]:
# Jawaban Soal 2

**Penjelasan:** …

### Soal 3 — Tipe data & missing value

In [ ]:
# Jawaban Soal 3

**Penjelasan:** …

### Soal 4 — Nilai tidak valid & outlier

In [ ]:
# Jawaban Soal 4

**Penjelasan:** …

### Soal 5 — EDA & insight

In [ ]:
# Jawaban Soal 5

**Penjelasan:** …